## Colab Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL  = "https://github.com/tackes/Modern-Time-Series-Forecasting-Cohort.git"
    REPO_PATH = "/content/packt-modern-time-series"

    if not os.path.exists(REPO_PATH):
        os.system(f"git clone -q {REPO_URL} {REPO_PATH}")

    os.chdir(f"{REPO_PATH}/student_notebooks")

    if REPO_PATH not in sys.path:
        sys.path.insert(0, REPO_PATH)

    os.system("pip install -q lightgbm mlforecast")

print(f"✓ Setup complete — {os.getcwd()}")

---

# Module 5 — ML Forecasting with LightGBM
**Type:** [Code With Me]  
**Time:** 45 minutes  
**Job:** Prove that a feature-engineered ML pipeline beats the statistical baselines. Quantify the compute cost. Make an honest recommendation about whether the improvement justifies the infrastructure.

---
## 5.1 — Setup
**[Watch Only]**

---

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['figure.figsize'] = (14, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from mlforecast.utils import PredictionIntervals

from config import (
    ARTIFACT_DIR,
    PARAMS_DIR,
    HORIZON,
    SEASON_LENGTH,
    N_WINDOWS,
    STEP_SIZE,
    REFIT,
    MICRO_SUBSET_N,
    WORKSHOP_SUBSET_N,
    RANDOM_SEED,
    USE_TUNED_PARAMS,
    USE_INTERVALS,
)
from src.checkpointing import load_checkpoint
from src.evaluation import score_forecasts
from src.schemas import validate_forecast
from src.workshop_utils import get_micro_subset
from src.forecast_schema import reshape_mlforecast_cv
from src.plotting import plot_forecast_overlay

print("Setup complete.")
print(f"USE_TUNED_PARAMS = {USE_TUNED_PARAMS}")
print(f"USE_INTERVALS    = {USE_INTERVALS}")

---
## 5.2 — Load Panel and Micro Subset
**[Watch Only]**

---

In [ ]:
panel = load_checkpoint("03_validated_panel")
micro, top_series = get_micro_subset(panel, n=MICRO_SUBSET_N)

print(f"Micro panel: {micro['unique_id'].nunique()} series, {len(micro):,} rows")

---
## 5.3 — Why ML Forecasting?
**[Watch Only]**

**LightGBM via MLForecast is a global model.** It trains on all series simultaneously — each series becomes a set of training rows. The model learns patterns across the full portfolio, not just within a single series.

**The compute tradeoff is real:** AutoETS on 1,000 series takes seconds. LightGBM takes longer and produces an artifact you must version, deploy, and monitor. The accuracy gain must justify that cost.

**The three feature configurations we will build (5.5A–5.5C):**

| Section | Feature type | What it adds |
|---|---|---|
| **5.5A — Base** | Lags (7, 14, 28), rolling mean, day-of-week + month | Demand history and calendar |
| **5.5B — Rich** | Extended lags, rolling stats, Fourier features, seasonal rolling | More complete numeric/date signal |
| **5.5C — Categorical** | M5 hierarchy labels (category, dept, state, store) as static features | Structural context no univariate model can see |



---
## 5.4 — Load LightGBM Parameters
**[Watch Only]**

---

In [ ]:
params_file = "mlforecast_lgb_tuned.json" if USE_TUNED_PARAMS else "mlforecast_lgb_default.json"
params_path = PARAMS_DIR / params_file

with open(params_path) as f:
    lgb_params = json.load(f)

lgb_params["random_state"] = RANDOM_SEED

print(f"Loaded {len(lgb_params)} params from: {params_file}")

---
## 5.5A — Configure MLForecast (Base)
**[Code With Me — 3 lines]**

Fill in `lags`, `lag_transforms`, and `date_features`. Use weekly, biweekly, and monthly lags. Rolling mean on lag 7 with a 28-day window. Day-of-week and month as date features.

Sections 5.5B and 5.5C define richer configs — you will compare all three in 5.9A and 5.9B.

---

In [ ]:
mlf = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=__FILL_IN__,                    # [7, 14, 28]
    lag_transforms={
        7: __FILL_IN__,                  # [RollingMean(window_size=28)]
    },
    date_features=__FILL_IN__,           # ["dayofweek", "month"]
)

print("MLForecast configured.")
print(f"  Lags          : {mlf.ts.lags}")
print(f"  Date features : {mlf.ts.date_features}")
print(f"  Model         : {next(iter(mlf.models.values())).__class__.__name__}")

---
### 5.5B — Feature Engineering Reference: Enhanced Numeric and Date Features
**[Watch Only]**

Feature families:
- **More lags** — demand echoes at 1, 2, 3, 4 weeks
- **Rolling statistics** — mean (level), std (volatility), min/max (range), quantiles (distributional shape)
- **Fourier features** — encode weekly and annual seasonality as sin/cos waves. Better than integer `dayofweek` on short or sparse series because the circular structure is baked into the input rather than learned from scratch.
- **Seasonal rolling mean** — average of the same weekday over the past N weeks; directly encodes the weekly profile

The 5.5A config is deliberately minimal. This block shows the full toolkit. 5.5C adds categorical hierarchy on top.

---

In [ ]:
import numpy as np
from mlforecast import MLForecast
from mlforecast.lag_transforms import (
    RollingMean, RollingStd, RollingMin, RollingMax,
    RollingQuantile, SeasonalRollingMean,
)

LAGS = [7, 14, 21, 28]

ROLLING_28 = [
    RollingMean(window_size=28),
    RollingStd(window_size=28),
    RollingMin(window_size=28),
    RollingMax(window_size=28),
    RollingQuantile(p=0.25, window_size=28),
    RollingQuantile(p=0.75, window_size=28),
]

SEASONAL_ROLLING = [
    SeasonalRollingMean(season_length=7, window_size=4),
    SeasonalRollingMean(season_length=7, window_size=8),
]

def fourier_sin1_weekly(dates): return np.sin(2 * np.pi * 1 * dates.dayofweek / 7)
def fourier_cos1_weekly(dates): return np.cos(2 * np.pi * 1 * dates.dayofweek / 7)
def fourier_sin2_weekly(dates): return np.sin(2 * np.pi * 2 * dates.dayofweek / 7)
def fourier_cos2_weekly(dates): return np.cos(2 * np.pi * 2 * dates.dayofweek / 7)
def fourier_sin1_annual(dates): return np.sin(2 * np.pi * 1 * dates.dayofyear / 365.25)
def fourier_cos1_annual(dates): return np.cos(2 * np.pi * 1 * dates.dayofyear / 365.25)

DATE_FEATURES = [
    "month",
    fourier_sin1_weekly, fourier_cos1_weekly,
    fourier_sin2_weekly, fourier_cos2_weekly,
    fourier_sin1_annual, fourier_cos1_annual,
]

mlf_rich = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=LAGS,
    lag_transforms={
        7:  ROLLING_28 + SEASONAL_ROLLING,
        14: [RollingMean(window_size=28)],
    },
    date_features=DATE_FEATURES,
)

print(f"Rich config: {len(LAGS)} lags, {len(ROLLING_28+SEASONAL_ROLLING)} lag-7 transforms, {len(DATE_FEATURES)} date features")

---
### 5.5C — Static Categorical Features: M5 Product Hierarchy
**[Watch Only]**

Every model so far — Naive, SeasonalNaive, AutoETS, Chronos — processes each series in isolation. LightGBM is the first model in our stack that can use structural information that *spans* series.

The M5 unique_id encodes a full product-store hierarchy. We extract it and pass it as **static features** — the same value for every row of a given series. MLForecast passes them to LightGBM as constant columns at every training row.

Key points:
- `static_features` is passed to `cross_validation()` as a keyword argument — not to the constructor
- LightGBM requires numeric inputs — we integer-encode each label with pandas `.cat.codes`
- In production, static features come from a product master table, not from parsing series IDs

---

In [ ]:
# M5 unique_id format: {CATEGORY}_{DEPT_NUM}_{ITEM_NUM}_{STATE_ID}_{STORE_NUM}
# Example: FOODS_2_360_WI_2 → cat=FOODS, dept=FOODS_2, state=WI, store=WI_2

def add_m5_categoricals(df: pd.DataFrame) -> pd.DataFrame:
    """Extract and integer-encode M5 hierarchy labels. LightGBM requires numeric inputs."""
    parts = df["unique_id"].str.split("_")
    df = df.copy()
    df["cat_id"]   = parts.str[0]
    df["dept_id"]  = parts.str[:2].str.join("_")
    df["state_id"] = parts.str[3]
    df["store_id"] = parts.str[3:5].str.join("_")
    for col in ["cat_id", "dept_id", "state_id", "store_id"]:
        df[col] = df[col].astype("category").cat.codes
    return df

STATIC_FEATURES = ["cat_id", "dept_id", "state_id", "store_id"]
micro_cat = add_m5_categoricals(micro)

# static_features is passed to cross_validation(), not to the constructor
mlf_cat = MLForecast(
    models=[lgb.LGBMRegressor(**lgb_params)],
    freq="D",
    lags=LAGS,
    lag_transforms={7: ROLLING_28 + SEASONAL_ROLLING, 14: [RollingMean(window_size=28)]},
    date_features=DATE_FEATURES,
)

print(f"micro_cat shape: {micro_cat.shape} — static cols: {STATIC_FEATURES}")
micro_cat[["unique_id"] + STATIC_FEATURES].drop_duplicates().head()

**Expected output:**
```
MLForecast configured.
  Lags          : [7, 14, 28]
  Date features : ['dayofweek', 'month']
  Model         : LGBMRegressor
```

---
## 5.6 — Run Cross-Validation on the Micro Subset
**[Watch Only]**

---

In [ ]:
%%time
# PredictionIntervals requires refit=True
cv_ml_micro = mlf_rich.cross_validation(
    df=micro,
    h=HORIZON,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=True if USE_INTERVALS else REFIT,
    prediction_intervals=PredictionIntervals(n_windows=N_WINDOWS, h=HORIZON) if USE_INTERVALS else None,
    level=[80] if USE_INTERVALS else None,
)

print(f"CV complete: {cv_ml_micro.shape[0]:,} rows × {cv_ml_micro.shape[1]} columns")
cv_ml_micro.head(3)

---
## 5.7 — Reshape MLForecast CV Output
**[Watch Only]**

`reshape_mlforecast_cv` (from `src.forecast_schema`) renames the wide MLForecast columns to the workshop schema. When `PredictionIntervals` is active, it picks up the native `LGBMRegressor-lo-80` / `LGBMRegressor-hi-80` columns — no residual fabrication. The wide format was shown at the end of 5.6.

In [ ]:
# cv_ml_micro came from mlf_rich.cross_validation() in 5.6 → model_name="LightGBM-Rich"
ml_micro = reshape_mlforecast_cv(
    cv_ml_micro,
    model_col="LGBMRegressor",
    stage="ml",
    model_name="LightGBM-Rich",
)

print(
    f"Reshaped: {ml_micro.shape[0]:,} rows | "
    f"model: {ml_micro['model'].unique()[0]}"
)
ml_micro.head(3)

**Expected output:**
```
Reshaped: 4,200 rows | model: LightGBM-Rich
```
Columns: `unique_id`, `ds`, `y`, `model`, `y_hat`, `lo_80`, `hi_80`, `cutoff`, `stage`

---
## 5.8 — Plot: LightGBM vs Best Baseline
**[Watch Only]**

In [ ]:
# Compare LightGBM-Rich vs SeasonalNaive on a single series
sample_uid = top_series[0]
sample_cut = ml_micro["cutoff"].unique()[-1]

try:
    baseline_micro = pd.read_parquet(ARTIFACT_DIR / "04_baseline_micro_forecasts.parquet")
    combined = pd.concat(
        [ml_micro, baseline_micro[baseline_micro["model"] == "SeasonalNaive"]],
        ignore_index=True,
    )
    models = ["LightGBM-Rich", "SeasonalNaive"]
except Exception:
    combined = ml_micro
    models = ["LightGBM-Rich"]

plot_forecast_overlay(
    actuals_df=panel,
    forecasts_df=combined,
    unique_id=sample_uid,
    cutoff=sample_cut,
    models=models,
    title=f"LightGBM-Rich vs SeasonalNaive — {sample_uid} (Window 3)",
)

---
## 5.9A — Micro Subset Comparison: Base vs Rich vs Categorical
**[Watch Only]**

This cell runs all three configs on the 50-series micro subset. The comparison table shows that 50 series is not enough to reliably distinguish three feature sets — the ordering here is likely noise. Section 5.9B shows the full-panel result, which is the one to use for decisions.

---

In [ ]:
%%time
# 5.9A — Micro Subset Comparison: Base vs Rich vs Categorical
# ─────────────────────────────────────────────────────────────
# mlf_rich already ran in 5.6. Reuse cv_ml_micro (reshape done in 5.7 → ml_micro).
# This cell additionally runs mlf (base) and mlf_cat on micro, then compares all three.

def _get_score(scores_df, metric):
    row = scores_df[scores_df["metric"] == metric]
    return row.iloc[0]["score"] if len(row) > 0 else float("nan")

# ── mlf_rich result from 5.6/5.7 — validate and score ────────────────────────
ml_rich_micro_val = validate_forecast(ml_micro, artifact_name="05_ml_rich_micro")
scores_rich = score_forecasts(ml_rich_micro_val, subset_name=f"micro_{MICRO_SUBSET_N}")

# ── Base config (mlf) on micro ────────────────────────────────────────────────
cv_base_micro = mlf.cross_validation(
    df=micro,
    h=HORIZON,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=True if USE_INTERVALS else REFIT,
    prediction_intervals=PredictionIntervals(n_windows=N_WINDOWS, h=HORIZON) if USE_INTERVALS else None,
    level=[80] if USE_INTERVALS else None,
)
ml_base_micro = reshape_mlforecast_cv(cv_base_micro, model_col="LGBMRegressor", stage="ml", model_name="LightGBM")
ml_base_micro_val = validate_forecast(ml_base_micro, artifact_name="05_ml_base_micro")
scores_base = score_forecasts(ml_base_micro_val, subset_name=f"micro_{MICRO_SUBSET_N}")

# ── Categorical config (mlf_cat) on micro ─────────────────────────────────────
cv_cat_micro = mlf_cat.cross_validation(
    df=micro_cat,
    h=HORIZON,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=True if USE_INTERVALS else REFIT,
    prediction_intervals=PredictionIntervals(n_windows=N_WINDOWS, h=HORIZON) if USE_INTERVALS else None,
    level=[80] if USE_INTERVALS else None,
    static_features=STATIC_FEATURES,
)
ml_cat_micro = reshape_mlforecast_cv(cv_cat_micro, model_col="LGBMRegressor", stage="ml", model_name="LightGBM-Cat")
ml_cat_micro_val = validate_forecast(ml_cat_micro, artifact_name="05_ml_cat_micro")
scores_cat = score_forecasts(ml_cat_micro_val, subset_name=f"micro_{MICRO_SUBSET_N}")

# ── Comparison table ───────────────────────────────────────────────────────────
W = 24
print(f"Micro Subset Comparison ({MICRO_SUBSET_N} series, {N_WINDOWS} windows):")
print(f"  {'Config':<{W}} {'wMAPE':>8}  {'Bias':>8}  {'IntervalScore_80':>16}")
print(f"  {'-'*66}")
for label, scores in [
    ("Base — 5.5A",        scores_base),
    ("Rich — 5.5B",        scores_rich),
    ("Categorical — 5.5C", scores_cat),
]:
    wmape  = _get_score(scores, "wMAPE")
    bias   = _get_score(scores, "Bias")
    iscore = _get_score(scores, "IntervalScore_80")
    print(f"  {label:<{W}} {wmape:>8.4f}  {bias:>+8.4f}  {iscore:>16.4f}")
print()
print(f"Key point: {MICRO_SUBSET_N} series is not enough to reliably distinguish these configs.")
print("Any ordering here is likely noise. See 5.9B for the full-panel view.")

---
## 5.9B — Full Panel Comparison: Base vs Rich vs Categorical
**[Watch Only]**

Load all three precomputed full-panel artifacts (1,000 series). Compare the ordering here with 5.9A — the full-panel result is the reliable one. Categorical features add structural context that no univariate model can access.

---

In [ ]:
# 5.9B — Full Panel Comparison: Base vs Rich vs Categorical (Precomputed)
# ─────────────────────────────────────────────────────────────────────────

try:
    ml_full_base = load_checkpoint("05_ml_forecasts")
    ml_full_rich = load_checkpoint("05_ml_rich_forecasts")
    ml_full_cat  = load_checkpoint("05_ml_cat_forecasts")

    subset_label     = f"workshop_{WORKSHOP_SUBSET_N}"
    scores_full_base = score_forecasts(ml_full_base, subset_name=subset_label)
    scores_full_rich = score_forecasts(ml_full_rich, subset_name=subset_label)
    scores_full_cat  = score_forecasts(ml_full_cat,  subset_name=subset_label)

    W = 24
    print(f"Full Panel Comparison ({WORKSHOP_SUBSET_N:,} series, {N_WINDOWS} windows — RELIABLE):")
    print(f"  {'Config':<{W}} {'wMAPE':>8}  {'Bias':>8}  {'IntervalScore_80':>16}")
    print(f"  {'-'*66}")
    for label, scores in [
        ("Base — 5.5A",        scores_full_base),
        ("Rich — 5.5B",        scores_full_rich),
        ("Categorical — 5.5C", scores_full_cat),
    ]:
        wmape  = _get_score(scores, "wMAPE")
        bias   = _get_score(scores, "Bias")
        iscore = _get_score(scores, "IntervalScore_80")
        print(f"  {label:<{W}} {wmape:>8.4f}  {bias:>+8.4f}  {iscore:>16.4f}")

    print()
    print("Key lesson: micro result is noise. Full panel is the signal.")
    print("Categorical features add structural context no univariate model can access.")

except FileNotFoundError as e:
    print(f"Artifact not found: {e}")
    print("Run: python src/build_offline_artifacts.py --stages ml ml_rich ml_cat")

**Expected output:**
```
Full Panel Comparison (1,000 series, 3 windows — RELIABLE):
  Config                  wMAPE      Bias  IntervalScore_80
  ────────────────────────────────────────────────────────
  Base — 5.5A            0.XXXX   +0.XXXX           XX.XXXX
  Rich — 5.5B            0.XXXX   +0.XXXX           XX.XXXX
  Categorical — 5.5C     0.XXXX   +0.XXXX           XX.XXXX
```

---
## 5.10 — Module Leaderboard: ML vs Baselines
**[Watch Only]**

In [ ]:
# 5.10 — Build the full leaderboard: all three ML configs + statistical baselines
# ─────────────────────────────────────────────────────────────────────────────
# Reuse artifacts already loaded in 5.9B. Load baselines fresh.

subset_label = f"workshop_{WORKSHOP_SUBSET_N}"

# Load baselines
baseline_full   = load_checkpoint("04_baseline_forecasts")
baseline_scores = score_forecasts(baseline_full, subset_name=subset_label)

# ML scores — reuse from 5.9B if in scope, otherwise reload
try:
    _base_ok = "scores_full_base" in dir() and scores_full_base is not None
except NameError:
    _base_ok = False

if not _base_ok:
    ml_full_base = load_checkpoint("05_ml_forecasts")
    ml_full_rich = load_checkpoint("05_ml_rich_forecasts")
    ml_full_cat  = load_checkpoint("05_ml_cat_forecasts")
    scores_full_base = score_forecasts(ml_full_base, subset_name=subset_label)
    scores_full_rich = score_forecasts(ml_full_rich, subset_name=subset_label)
    scores_full_cat  = score_forecasts(ml_full_cat,  subset_name=subset_label)

from src.evaluation import build_leaderboard
leaderboard_5 = build_leaderboard([baseline_scores, scores_full_base, scores_full_rich, scores_full_cat])

display_cols   = ["model", "wMAPE", "Bias", "IntervalScore_80"]
available_cols = [c for c in display_cols if c in leaderboard_5.columns]
wmape_lb = (
    leaderboard_5[available_cols]
    .dropna(subset=["wMAPE"])
    .sort_values("wMAPE")
)
wmape_lb

**Expected output:** Leaderboard sorted by wMAPE — three LightGBM variants at the top, then baselines.
```
Updated leaderboard — all models (sorted by wMAPE):
          model    wMAPE      Bias
   LightGBM-Cat   0.XXXX   +0.XXXX
  LightGBM-Rich   0.XXXX   +0.XXXX
       LightGBM   0.XXXX   +0.XXXX
        AutoETS   0.XXXX   +0.XXXX
Chronos-t5-mini   0.XXXX   -0.XXXX
  SeasonalNaive   0.XXXX   -0.XXXX
          Naive   0.XXXX   +0.XXXX
```

---
## 5.11 — Save Micro ML Artifacts
**[Watch Only]**

Save all three micro artifacts for optional take-home analysis. The full-subset artifacts were already loaded from precomputed checkpoints.

---

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

micro_artifacts = {
    "05_ml_base_micro_forecasts.parquet": ml_base_micro_val,
    "05_ml_rich_micro_forecasts.parquet": ml_rich_micro_val,
    "05_ml_cat_micro_forecasts.parquet":  ml_cat_micro_val,
}
for fname, df_val in micro_artifacts.items():
    path = ARTIFACT_DIR / fname
    df_val.to_parquet(path, index=False)
    print(f"  ✓ Micro artifact saved : {fname} ({len(df_val):,} rows)")

print()
print("Full-panel artifacts (precomputed checkpoints):")
print(f"  ✓ 05_ml_forecasts.parquet          ({len(ml_full_base):,} rows) — LightGBM base")
print(f"  ✓ 05_ml_rich_forecasts.parquet      ({len(ml_full_rich):,} rows) — LightGBM-Rich")
print(f"  ✓ 05_ml_cat_forecasts.parquet       ({len(ml_full_cat):,} rows)  — LightGBM-Cat")